# Figure 1C, D, E — Lab AUC Violin Plots

Violin plots of log2 post-treatment lab values by AE grade.

**Prerequisites:** Run `generate_lab_cache.py` first to create `lab_results_cache.pkl`

**Outputs per panel:** PDF, PNG, and CSV

In [ ]:
# ---------------------------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------------------------
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import mannwhitneyu, norm

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    print(f"Warning: Arial not found, using {_arial_path}")
else:
    print(f"Arial resolved to: {_arial_path}")

In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
FIG_DIR = RESULTS / "main"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = DATA / "lab_results_cache.pkl"
assert CACHE_PATH.exists(), f"Missing: {CACHE_PATH.name} (run generate_lab_cache.py first)"

print(f"Cache: {CACHE_PATH.name}")
print(f"Output folder: {FIG_DIR.name}")

In [ ]:
# ---------------------------------------------------------------------------
# LOAD CACHE
# ---------------------------------------------------------------------------
with open(CACHE_PATH, 'rb') as f:
    results_df = pickle.load(f)

print(f"Loaded {len(results_df):,} records")
print(f"\nAvailable ae_type -> lab_test:")
for ae_type, sub in results_df.groupby('ae_type'):
    print(f"  {ae_type}: {sorted(sub['lab_test'].unique().tolist())}")

print(f"\nlog2_post availability by grade:")
for grade in sorted(results_df['grade'].unique()):
    g = results_df[results_df['grade'] == grade]
    valid = g['log2_post'].notna().sum()
    print(f"  Grade {int(grade)}: {valid:,}/{len(g):,} ({100*valid/len(g):.1f}%)")

In [ ]:
# ---------------------------------------------------------------------------
# PLOTTING CONFIG
# ---------------------------------------------------------------------------
MIN_N = 5
GRADE_GRAY = '#95a5a6'

AE_CONFIGS = [
    {
        "panel": "1C",
        "ae_type": "adrenal_insufficiency",
        "lab_test": "Cortisol",
        "ylabel": "Log2 Cortisol AUC",
        "out_pdf": "Lab_Violin_Adrenal_Insuff_1C.pdf",
        "out_png": "Lab_Violin_Adrenal_Insuff_1C.png",
        "out_csv": "Lab_Violin_Adrenal_Insuff_1C_stats.csv",
    },
    {
        "panel": "1D",
        "ae_type": "hypothyroidism",
        "lab_test": "TSH",
        "ylabel": "Log2 TSH AUC",
        "out_pdf": "Lab_Violin_Hypothyroidism_1D.pdf",
        "out_png": "Lab_Violin_Hypothyroidism_1D.png",
        "out_csv": "Lab_Violin_Hypothyroidism_1D_stats.csv",
    },
    {
        "panel": "1E",
        "ae_type": "liver_toxicity",
        "lab_test": "ALT",
        "ylabel": "Log2 ALT AUC",
        "out_pdf": "Lab_Violin_Liver_Tox_1E.pdf",
        "out_png": "Lab_Violin_Liver_Tox_1E.png",
        "out_csv": "Lab_Violin_Liver_Tox_1E_stats.csv",
    },
]

def get_grade_colors(grades):
    """Gray for grade 0; light -> dark rocket_r for grade 1+"""
    cmap = sns.color_palette("rocket_r", as_cmap=True)
    non_zero = sorted(g for g in grades if g != 0)
    n = len(non_zero)
    shades = {}
    if n > 0:
        t_vals = np.linspace(0.25, 0.85, n) if n > 1 else [0.55]
        for g, t in zip(non_zero, t_vals):
            shades[g] = cmap(t)
    return {0: GRADE_GRAY, **shades}

print("Config loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------
def group_low_count_grades(data_by_grade, min_n=5):
    """Merge grades with fewer than min_n samples with adjacent grades."""
    if not data_by_grade:
        return {}
    numeric = {}
    for k, v in data_by_grade.items():
        try:
            numeric[float(k)] = v
        except (ValueError, TypeError):
            continue
    if not numeric:
        return {}
    grades = sorted(numeric.keys())
    if not any(len(numeric[g]) < min_n for g in grades if g != 0.0):
        return numeric
    grouped, i = {}, 0
    while i < len(grades):
        g = grades[i]
        d = numeric[g]
        if g == 0.0:
            grouped[0.0] = d; i += 1; continue
        if len(d) < min_n and i + 1 < len(grades):
            ng = grades[i + 1]
            grouped[(g + ng) / 2] = np.concatenate([d, numeric[ng]])
            i += 2
        elif len(d) < min_n and i > 0:
            pg = grades[i - 1]
            if pg != 0.0 and pg in grouped:
                grouped[pg] = np.concatenate([grouped[pg], d])
            elif pg != 0.0:
                grouped[(pg + g) / 2] = np.concatenate([numeric[pg], d])
            else:
                grouped[g] = d
            i += 1
        else:
            grouped[g] = d; i += 1
    return grouped

def grade_label(grade):
    if grade == 0.0:
        return 'No AE'
    if grade == int(grade):
        return f'Grade {int(grade)}'
    return f'Grade {int(grade - 0.5)}-{int(grade + 0.5)}'

def jonckheere_terpstra(groups):
    """Jonckheere-Terpstra trend test."""
    k = len(groups)
    n_total = sum(len(g) for g in groups)
    JT = 0
    for i in range(k):
        for j in range(i + 1, k):
            gi, gj = np.asarray(groups[i]), np.asarray(groups[j])
            U = int((gj[:, None] > gi[None, :]).sum()) if len(gi) and len(gj) else 0
            JT += U
    n_i = np.array([len(g) for g in groups])
    mean_JT = (n_total**2 - np.sum(n_i**2)) / 4
    var_JT = (n_total**2 * (2 * n_total + 3) - np.sum(n_i**2 * (2 * n_i + 3))) / 72
    if var_JT <= 0:
        return JT, np.nan, np.nan
    z = (JT - mean_JT) / np.sqrt(var_JT)
    log_p = np.log(2) + norm.logsf(abs(z))
    p = np.exp(log_p) if log_p > -700 else 0.0
    return JT, z, p

print("Helper functions defined.")

In [ ]:
# ---------------------------------------------------------------------------
# MAIN PLOTTING FUNCTION
# ---------------------------------------------------------------------------
def process_ae(cfg, results_df):
    ae_type, lab_test, ylabel = cfg["ae_type"], cfg["lab_test"], cfg["ylabel"]
    pdf_path = FIG_DIR / cfg["out_pdf"]
    png_path = FIG_DIR / cfg["out_png"]
    csv_path = FIG_DIR / cfg["out_csv"]

    sub = results_df[(results_df['ae_type'] == ae_type) & (results_df['lab_test'] == lab_test)].copy()
    print(f"\n=== [{cfg['panel']}] {ae_type} / {lab_test} -- total rows: {len(sub):,} ===")

    # Collect both log2 and raw values by grade
    data_by_grade = {}
    raw_by_grade = {}
    for g in sub['grade'].unique():
        log2_vals = sub.loc[sub['grade'] == g, 'log2_post'].dropna().values
        raw_vals = sub.loc[sub['grade'] == g, 'post_auc'].dropna().values
        if len(log2_vals) > 0:
            data_by_grade[float(g)] = log2_vals
            raw_by_grade[float(g)] = raw_vals

    grouped = group_low_count_grades(data_by_grade, min_n=MIN_N)
    raw_grouped = group_low_count_grades(raw_by_grade, min_n=MIN_N)
    if not grouped:
        print(f"  WARNING: no data -- skipping.")
        return

    grades = sorted(grouped.keys())

    # Adjacent-grade Mann-Whitney p-values
    p_vals = [None]
    for i in range(1, len(grades)):
        a, b = grouped[grades[i - 1]], grouped[grades[i]]
        if len(a) >= 3 and len(b) >= 3:
            try:
                p_vals.append(mannwhitneyu(a, b, alternative='two-sided').pvalue)
            except Exception:
                p_vals.append(None)
        else:
            p_vals.append(None)

    grade_colors = get_grade_colors(grades)

    labels, data_lists, colors = [], [], []
    for g in grades:
        d = grouped[g]
        labels.append(f'{grade_label(g)}\nn={len(d):,}')
        data_lists.append(d)
        colors.append(grade_colors.get(g, GRADE_GRAY))

    jt_stat, jt_z, jt_p = jonckheere_terpstra(data_lists)
    jt_ptxt = 'Trend p<0.001' if jt_p < 0.001 else f'Trend p={jt_p:.3f}'
    print(f"  Jonckheere-Terpstra: JT={jt_stat}, z={jt_z:.3f}, p={jt_p:.4g}")

    # Stats rows for CSV (includes both log2 and raw values)
    stat_rows = []
    for i, g in enumerate(grades):
        d = grouped[g]  # log2 values
        r = raw_grouped[g]  # raw values
        stat_rows.append({
            "panel": cfg["panel"], "ae_type": ae_type, "lab_test": lab_test,
            "grade": grade_label(g), "n": len(d),
            # Log2 stats (used for plotting)
            "log2_median": float(np.median(d)),
            "log2_q1": float(np.percentile(d, 25)), 
            "log2_q3": float(np.percentile(d, 75)),
            # Raw stats
            "raw_median": float(np.median(r)),
            "raw_q1": float(np.percentile(r, 25)), 
            "raw_q3": float(np.percentile(r, 75)),
            "raw_mean": float(np.mean(r)),
            "raw_std": float(np.std(r)),
            # P-values
            "p_vs_prev_grade": p_vals[i],
            "trend_JT_stat": jt_stat, "trend_z": jt_z, "trend_p": jt_p,
        })
    
    # Save CSV for this panel
    stats_df = pd.DataFrame(stat_rows)
    stats_df.to_csv(csv_path, index=False)
    print(f"  Saved CSV: {csv_path.name}")

    # Violin plot
    fig, ax = plt.subplots(figsize=(2.8, 1.9), constrained_layout=True)
    positions = list(range(len(grades)))

    parts = ax.violinplot(data_lists, positions=positions, showmeans=False,
                          showmedians=False, showextrema=False, widths=0.6)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c); pc.set_alpha(0.35)
        pc.set_edgecolor(c); pc.set_linewidth(1.2)

    np.random.seed(42)
    for pos, d, c in zip(positions, data_lists, colors):
        if len(d) > 0:
            jitter = np.random.normal(pos, 0.04, len(d))
            ax.scatter(jitter, d, alpha=0.4, s=8, color=c,
                       edgecolors='white', linewidths=0.2, zorder=3)
            ax.hlines(np.median(d), pos - 0.22, pos + 0.22,
                      colors='black', linewidth=1.6, zorder=4)

    all_vals = np.concatenate(data_lists)
    lo, hi = float(all_vals.min()), float(all_vals.max())
    span = (hi - lo) if hi > lo else (abs(hi) + 1.0)
    base = hi + span * 0.07
    tick = span * 0.022

    for i in range(1, len(grades)):
        if p_vals[i] is None:
            continue
        x1, x2 = positions[i - 1] + 0.07, positions[i] - 0.07
        ax.plot([x1, x1, x2, x2], [base - tick, base, base, base - tick],
                color='black', linewidth=0.9, zorder=5)
        ptxt = 'p<0.001' if p_vals[i] < 0.001 else f'p={p_vals[i]:.3f}'
        ax.text((x1 + x2) / 2, base + span * 0.012, ptxt,
                ha='center', va='bottom', fontsize=5, zorder=5)

    ax.set_ylabel(ylabel, fontsize=7)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=6)
    ax.tick_params(axis='y', labelsize=6)
    ax.set_ylim(lo - span * 0.18, base + span * 0.14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.text(0.02, 1.00, jt_ptxt, transform=ax.transAxes,
            ha='left', va='top', fontsize=5, style='italic')

    # Save PDF
    with PdfPages(str(pdf_path)) as pdf:
        pdf.savefig(fig, dpi=450)
    print(f"  Saved PDF: {pdf_path.name}")
    
    # Save PNG
    fig.savefig(png_path, dpi=450, bbox_inches='tight')
    print(f"  Saved PNG: {png_path.name}")
    
    plt.show()

print("process_ae() defined.")

In [ ]:
# ---------------------------------------------------------------------------
# RUN ALL PANELS
# ---------------------------------------------------------------------------
print("="*80)
print("GENERATING FIGURES")
print("="*80)

for cfg in AE_CONFIGS:
    process_ae(cfg, results_df)

print("\n" + "="*80)
print("DONE - All outputs saved to results/main/")
print("="*80)